# HemaFAIR OMOP — Query Notebook

This notebook connects to your **completed** OMOP database (`trainee_XX_answers`) and lets you explore the data with pre-built queries.

No ETL needed — the data is already loaded. Just run the cells in order.

**What's in here:**
- Row count overview across all clinical tables
- Q1: How many patients have each diagnosis?
- Q2: What is the median ferritin level by sex?
- Q3: Transfusion status distribution per diagnosis
- Q4: Most common comorbidities (% of all patients)
- Q5: Patients on chelation with no transfusion history
- Q6: Measurement completeness

## Connection Settings

Replace `trainee_01` with your trainee number. The database is `trainee_XX_answers` — the completed version.

In [ ]:
DB_HOST     = "localhost"
DB_PORT     = 5432
DB_NAME     = "trainee_01_answers"  # replace with your trainee number
DB_USER     = "trainee_01"          # replace with your trainee number
DB_PASSWORD = "01"                   # replace with your trainee number

print(f"Connecting to {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

In [ ]:
import psycopg2
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

con = psycopg2.connect(
    host=DB_HOST, port=DB_PORT,
    dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD
)
con.autocommit = True

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{quote_plus(DB_PASSWORD)}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

def query(sql):
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn)

print("Connected.")

---
## Final Overview

Row counts across all clinical tables. All should be non-zero.

In [ ]:
query("""
SELECT 'person'               AS table_name, COUNT(*) AS row_count FROM omop.person
UNION ALL
SELECT 'observation_period',               COUNT(*) FROM omop.observation_period
UNION ALL
SELECT 'visit_occurrence',                 COUNT(*) FROM omop.visit_occurrence
UNION ALL
SELECT 'condition_occurrence',             COUNT(*) FROM omop.condition_occurrence
UNION ALL
SELECT 'measurement',                      COUNT(*) FROM omop.measurement
UNION ALL
SELECT 'observation',                      COUNT(*) FROM omop.observation
UNION ALL
SELECT 'procedure_occurrence',             COUNT(*) FROM omop.procedure_occurrence
UNION ALL
SELECT 'drug_exposure',                    COUNT(*) FROM omop.drug_exposure
ORDER BY table_name
""")

---
### Question 1: How many patients have each diagnosis?

Counts distinct patients per primary diagnosis — filtering by `condition_concept_id` and joining the `concept` table for the label.
This is how OMOP queries work in practice: filter by concept ID, not by source text.

In [ ]:
query("""
SELECT
    con.concept_name                AS diagnosis,
    COUNT(DISTINCT c.person_id)     AS n_patients
FROM omop.condition_occurrence c
JOIN omop.concept con ON con.concept_id = c.condition_concept_id
WHERE c.condition_concept_id IN (4287844, 4278669, 25518)  -- Alpha-thal, Beta-thal, Sickle cell
GROUP BY 1
ORDER BY 2 DESC;
""")

---
### Question 2: What is the median ferritin level by sex?

Filters by `measurement_concept_id` (LOINC concept for ferritin in serum), not by source value string.

In [ ]:
query("""
SELECT
    p.gender_source_value                                                           AS sex,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY m.value_as_number)::numeric, 1)
                                                                                    AS median_ferritin,
    COUNT(*)                                                                        AS n_measurements
FROM omop.measurement m
JOIN omop.person p ON p.person_id = m.person_id
WHERE m.measurement_concept_id = 37208753   -- LOINC: Ferritin in serum
  AND m.value_as_number IS NOT NULL
GROUP BY 1;
""")

---
### Question 3: What is the transfusion status distribution for each diagnosis?

Joins `condition_occurrence` (diagnosis) with `observation` (transfusion status) on `person_id`.

In [ ]:
query("""
SELECT
    con.concept_name                AS diagnosis,
    o.value_source_value            AS transfusion_status,
    COUNT(DISTINCT c.person_id)     AS n_patients
FROM omop.condition_occurrence c
JOIN omop.concept con ON con.concept_id = c.condition_concept_id
JOIN omop.observation o
    ON o.person_id = c.person_id
   AND o.observation_concept_id = 40758326  -- Transfusion status qualitative
WHERE c.condition_concept_id IN (4287844, 4278669, 25518)
  AND o.value_source_value IS NOT NULL
GROUP BY 1, 2
ORDER BY 1, 2;
""")

---
### Question 4: What are the most common comorbidities?

Excludes the three primary diagnoses and unmapped concepts (`concept_id = 0`), then counts how many patients have each comorbidity.

In [ ]:
total = query("SELECT COUNT(*) AS n FROM omop.person").iloc[0]['n']

query(f"""
SELECT
    con.concept_name                                           AS comorbidity,
    COUNT(DISTINCT co.person_id)                              AS n_patients,
    ROUND(COUNT(DISTINCT co.person_id) * 100.0 / {total}, 1) AS pct
FROM omop.condition_occurrence co
JOIN omop.concept con ON con.concept_id = co.condition_concept_id
WHERE co.condition_concept_id NOT IN (4287844, 4278669, 25518)
  AND co.condition_concept_id != 0
GROUP BY con.concept_name
ORDER BY n_patients DESC;
""")

---
### Question 5: Which patients are on chelation but have no transfusion history?

Joins `procedure_occurrence` (chelation) with `observation` (transfusion status = No).

In [ ]:
query("""
SELECT COUNT(DISTINCT pr.person_id) AS n_chelation_no_transfusion
FROM omop.procedure_occurrence pr
JOIN omop.observation o
    ON o.person_id = pr.person_id
   AND o.observation_concept_id = 40758326  -- Transfusion status qualitative
WHERE pr.procedure_concept_id = 4068544     -- Chelation treatment
  AND o.value_source_value = 'No';
""")

---
### Question 6: How complete is the measurement data?

For each numeric measurement, reports how many patients have a recorded value and how many are missing.

In [ ]:
total = query("SELECT COUNT(*) AS n FROM omop.person").iloc[0]['n']

query(f"""
SELECT
    COALESCE(NULLIF(con.concept_name, ''), m.value_source_value)   AS measurement,
    COUNT(DISTINCT m.person_id)                                     AS n_with_value,
    {total} - COUNT(DISTINCT m.person_id)                          AS n_missing,
    ROUND(COUNT(DISTINCT m.person_id) * 100.0 / {total}, 1)       AS pct_complete
FROM omop.measurement m
LEFT JOIN omop.concept con ON con.concept_id = m.measurement_concept_id
WHERE m.value_as_number IS NOT NULL
GROUP BY 1
ORDER BY n_with_value DESC;
""")